# LSTM Autoencoder: Encoding and Sampling SMILES

This notebook demonstrates:
1. Loading a trained LSTM Autoencoder model
2. Encoding SMILES strings to latent vectors
3. Sampling from the latent space
4. Reconstructing molecules
5. Interpolating between molecules


## 1. Setup and Imports


In [1]:
%load_ext autoreload
%autoreload 2
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Import deepchemography modules
from deepchemography import LSTMAutoencoder, OneHotVocab

print("Imports successful!")


Imports successful!


## 2. Load Trained Autoencoder Model


In [2]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [3]:
# Load model components
model_dir = Path('../models/autoencoder_converted/')

config = torch.load(model_dir / 'config.pt', weights_only=False)
vocab = torch.load(model_dir / 'vocab.pt', weights_only=False)
model_state = torch.load(model_dir / 'model_best.pt', weights_only=False)

print(f"Loaded checkpoint components from {model_dir}")


Loaded checkpoint components from ../models/autoencoder_converted


In [4]:
# Initialize and load the model
model = LSTMAutoencoder(vocab, config)
model.load_state_dict(model_state)
model = model.to(device)
model.eval()

print(f"✓ Model loaded successfully!")
print(f"  Vocabulary size: {len(vocab)}")
print(f"  Latent dimension: {config.d_z}")
print(f"  Encoder: {config.q_cell.upper()}, {config.q_n_layers} layers, {config.q_d_h} units")
print(f"  Decoder: {config.d_cell.upper()}, {config.d_n_layers} layers, {config.d_d_h} units")
print(f"  Batch normalization: {config.use_batch_norm}")


✓ Model loaded successfully!
  Vocabulary size: 30
  Latent dimension: 256
  Encoder: LSTM, 2 layers, 128 units
  Decoder: LSTM, 2 layers, 256 units
  Batch normalization: True


/data/aorlov/deepchemography/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:827: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


## 3. Define Helper Functions


In [6]:
def encode_smiles(smiles_list, model, batch_size=32):
    """
    Encode a list of SMILES strings to latent vectors.
    
    Args:
        smiles_list: List of SMILES strings
        model: Trained LSTMAutoencoder model
        batch_size: Batch size for processing
    
    Returns:
        numpy array of latent vectors (n_samples, latent_dim)
    """
    model.eval()
    latent_vectors = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(smiles_list), batch_size), desc="Encoding SMILES"):
            batch_smiles = smiles_list[i:i+batch_size]
            
            # Convert SMILES to tensors
            batch_tensors = []
            for smiles in batch_smiles:
                try:
                    tensor = model.string2tensor(smiles, device=device)
                    batch_tensors.append(tensor)
                except Exception as e:
                    print(f"Error processing SMILES '{smiles}': {e}")
                    # Fallback to simple carbon
                    tensor = model.string2tensor("C", device=device)
                    batch_tensors.append(tensor)
            
            # Encode to latent space
            z = model.forward_encoder(batch_tensors)
            latent_vectors.append(z.cpu().numpy())
    
    return np.vstack(latent_vectors)

print("✓ encode_smiles() function defined")


✓ encode_smiles() function defined


In [7]:
def sample_from_latent(model, z=None, n_samples=10, latent_std=1.0, max_len=100, temp=1.0, decode='greedy'):
    """
    Sample SMILES from the autoencoder latent space.
    
    Args:
        model: Trained LSTMAutoencoder model
        z: Optional latent vectors (numpy array or torch tensor). If None, samples from Gaussian
        n_samples: Number of samples to generate (only used if z is None)
        latent_std: Standard deviation for Gaussian sampling (only used if z is None)
        max_len: Maximum length of generated SMILES
        temp: Temperature for sampling (higher = more random, lower = more deterministic)
        decode: 'greedy' for deterministic, 'sample' for stochastic
    
    Returns:
        List of generated SMILES strings
    """
    model.eval()
    
    # If no latent vectors provided, sample from Gaussian prior
    if z is None:
        latent_dim = config.d_z
        z = torch.randn(n_samples, latent_dim, device=device) * latent_std
    elif isinstance(z, np.ndarray):
        z = torch.tensor(z, dtype=torch.float32, device=device)
    
    # Generate samples using the model's sample method
    samples = model.sample(n_batch=z.shape[0], max_len=max_len, z=z, temp=temp, decode=decode)
    
    return samples

print("✓ sample_from_latent() function defined")


✓ sample_from_latent() function defined


## 4. Example: Reconstruct SMILES (Encode → Decode)


In [8]:
# Test reconstruction on known drug molecules
test_smiles = [
    "CC(C)Cc1ccc(C(C)C(=O)O)cc1",     # Ibuprofen
    "CC(=O)Oc1ccccc1C(=O)O",           # Aspirin
    "CC(=O)Nc1ccc(O)cc1",              # Paracetamol
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",    # Caffeine
    "CC(C)NCC(O)COc1ccccc1",           # Propranolol
]

print("Test molecules for reconstruction:\\n")
for i, smiles in enumerate(test_smiles, 1):
    print(f"{i}. {smiles}")


Test molecules for reconstruction:\n
1. CC(C)Cc1ccc(C(C)C(=O)O)cc1
2. CC(=O)Oc1ccccc1C(=O)O
3. CC(=O)Nc1ccc(O)cc1
4. CN1C=NC2=C1C(=O)N(C(=O)N2C)C
5. CC(C)NCC(O)COc1ccccc1


In [9]:
# Encode test molecules
z_test = encode_smiles(test_smiles, model, batch_size=len(test_smiles))

# Decode with low temperature for deterministic reconstruction
reconstructed = sample_from_latent(model, z=z_test, temp=0.1, decode='greedy')

print("\\n" + "="*80)
print("Reconstruction Results (temp=0.1 for deterministic decoding)")
print("="*80 + "\\n")

for i, (original, recon) in enumerate(zip(test_smiles, reconstructed), 1):
    match = "✓" if original == recon else "✗"
    print(f"{i}. {match}")
    print(f"   Original:      {original}")
    print(f"   Reconstructed: {recon}")
    print()


Encoding SMILES: 100%|███████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.45s/it]


\n================================================================================
Reconstruction Results (temp=0.1 for deterministic decoding)
================================================================================\n
1. ✓
   Original:      CC(C)Cc1ccc(C(C)C(=O)O)cc1
   Reconstructed: CC(C)Cc1ccc(C(C)C(=O)O)cc1

2. ✓
   Original:      CC(=O)Oc1ccccc1C(=O)O
   Reconstructed: CC(=O)Oc1ccccc1C(=O)O

3. ✗
   Original:      CC(=O)Nc1ccc(O)cc1
   Reconstructed: CC(=O)c1ccc(O)cc1

4. ✓
   Original:      CN1C=NC2=C1C(=O)N(C(=O)N2C)C
   Reconstructed: CN1C=NC2=C1C(=O)N(C(=O)N2C)C

5. ✓
   Original:      CC(C)NCC(O)COc1ccccc1
   Reconstructed: CC(C)NCC(O)COc1ccccc1



## 5. Example: Sample from Gaussian Prior


In [ ]:
# Sample from Gaussian prior (mean=0, std=1.0)
n_samples = 10
samples = sample_from_latent(model, z=None, n_samples=n_samples, latent_std=1.0, temp=1.0)

print(f"Generated {len(samples)} SMILES from Gaussian prior:\\n")
for i, smiles in enumerate(samples, 1):
    print(f"{i:2d}. {smiles}")


## 6. Example: Interpolate Between Molecules


In [ ]:
# Interpolate between two molecules
smiles_1 = "CC(=O)Nc1ccc(O)cc1"  # Paracetamol
smiles_2 = "CC(=O)Oc1ccccc1C(=O)O"  # Aspirin

# Encode both molecules
z1 = encode_smiles([smiles_1], model, batch_size=1)
z2 = encode_smiles([smiles_2], model, batch_size=1)

print(f"Molecule 1: {smiles_1}")
print(f"Molecule 2: {smiles_2}")
print(f"\\nLatent vector distance: {np.linalg.norm(z1 - z2):.4f}")


In [ ]:
# Linear interpolation in latent space
n_steps = 10
alphas = np.linspace(0, 1, n_steps)

z_interp = []
for alpha in alphas:
    z = (1 - alpha) * z1 + alpha * z2
    z_interp.append(z)

z_interp = np.vstack(z_interp)

# Decode interpolated latent vectors
interpolated_smiles = sample_from_latent(model, z=z_interp, temp=0.1, decode='greedy')

print(f"\\nInterpolation from Molecule 1 to Molecule 2 ({n_steps} steps):\\n")
for i, (alpha, smiles) in enumerate(zip(alphas, interpolated_smiles)):
    print(f"α={alpha:.2f} | {smiles}")


## 7. Example: Explore Latent Space Neighborhoods


In [ ]:
# Pick a base molecule and explore its neighborhood
base_smiles = "CC(=O)Oc1ccccc1C(=O)O"  # Aspirin
z_base = encode_smiles([base_smiles], model, batch_size=1)

print(f"Base molecule: {base_smiles}\\n")
print("Exploring latent space neighborhood with different noise scales:\\n")

# Try different noise scales
for noise_scale in [0.05, 0.1, 0.2]:
    n_neighbors = 5
    z_neighbors = z_base + np.random.randn(n_neighbors, z_base.shape[1]) * noise_scale
    
    neighbors = sample_from_latent(model, z=z_neighbors, temp=0.5)
    
    print(f"Noise scale = {noise_scale}:")
    for i, neighbor in enumerate(neighbors, 1):
        print(f"  {i}. {neighbor}")
    print()
